In [1]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import torch
import pandas as pd
import plotly.express as px
from sklearn.preprocessing import MinMaxScaler
from collections import defaultdict
from itertools import product
import colorsys

In [2]:
# Path to your .ply file
file_path = '/storage/user/ayu/repos/HI-SLAM2/outputs/semantic/room0_24/3dgs_final.ply'

# Define the expected fields and corresponding NumPy dtypes
fields = [
    'x', 'y', 'z',
    'nx', 'ny', 'nz',
    'f_dc_0', 'f_dc_1', 'f_dc_2',
    'opacity',
    'ins_feat_0', 'ins_feat_1', 'ins_feat_2',
    'ins_feat_3', 'ins_feat_4', 'ins_feat_5',
    'scale_0', 'scale_1', 'scale_2',
    'rot_0', 'rot_1', 'rot_2', 'rot_3'
]

dtype = np.dtype([(name, 'f4') for name in fields])

# Read header and find vertex count and byte offset
with open(file_path, 'rb') as f:
    header_lines = []
    while True:
        line = f.readline()
        header_lines.append(line)
        if line.strip() == b'end_header':
            break

    # Decode and search for vertex count
    vertex_count = 0
    for line in header_lines:
        if line.startswith(b'element vertex'):
            vertex_count = int(line.decode().split()[-1])
            break

    # Sanity check
    if vertex_count == 0:
        raise ValueError("Could not determine number of vertices from header.")

    # Read binary data
    data = np.fromfile(f, dtype=dtype, count=vertex_count)

# Structured array is now loaded
array_2d = np.vstack([data[field] for field in fields]).T
print(f"Loaded {len(data)} vertices.")
print(array_2d[0])
pos = torch.Tensor(array_2d[:, :3])  # Extract position data (x, y, z)
normals = torch.Tensor(array_2d[:, 3:6])  # Extract normal data
spherical_harmonics = torch.Tensor(array_2d[:, 6:9])  # Extract spherical harmonics data
opacity = torch.Tensor(array_2d[:, 9])  # Extract opacity data
ins_feats = torch.Tensor(array_2d[:, 10:16])
scale = torch.Tensor(array_2d[:, 16:19])  # Extract scale data
rot = torch.Tensor(array_2d[:, 19:23])  # Extract rotation data

Loaded 148773 vertices.
[ 1.8541037  -1.2808143   2.5088077   0.          0.          0.
  1.2999976   1.014444    0.9874886   9.21066     0.38520512  0.5819982
 -0.19963631  1.1430593  -0.2525419  -0.27265152 -3.8380456  -4.005095
 -3.2408166   0.6347146  -0.15497819 -0.536697    0.41201517]


In [ ]:
from hislam2.gaussian.utils.post_processing import create_clusters_iterative
with torch.no_grad():
    # Create clusters iteratively
    clusters = create_clusters_iterative(ins_feats, pos, 7, 128, verbose=True)

2025-08-10 17:19:53,444 - root - INFO - Starting over segmentation with 128 initial clusters.
2025-08-10 17:19:53,766 - root - INFO - Creating clusters with similarity threshold 0.25 and resolution 32.
2025-08-10 17:19:53,787 - root - INFO - Building adjacency matrix...
2025-08-10 17:20:15,969 - root - INFO - Found 127 merged clusters.
2025-08-10 17:20:15,972 - root - INFO - Initial clustering done with 128 clusters.
2025-08-10 17:20:15,972 - root - INFO - Creating clusters with similarity threshold 0.5 and resolution 64.
2025-08-10 17:20:15,997 - root - INFO - Building adjacency matrix...
2025-08-10 17:21:56,251 - root - INFO - Found 118 merged clusters.
2025-08-10 17:21:56,253 - root - INFO - Iteration 1: Clustering done with resolution 64 and similarity threshold 0.5.
2025-08-10 17:21:56,253 - root - INFO - Number of clusters: 118.0
2025-08-10 17:21:56,254 - root - INFO - Creating clusters with similarity threshold 0.75 and resolution 64.
2025-08-10 17:21:56,268 - root - INFO - Buil